In [2]:
# 12 — Cell 1: BanglaBERT GroupKFold(5) setup (same settings as 09)
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"
OUT = f"{R}/bb_groupkfold"; os.makedirs(OUT, exist_ok=True)
MODEL, MAXLEN, SEED = "csebuetnlp/banglabert", 256, 0

try:
    from normalizer import normalize as bn_normalize
except ImportError:
    bn_normalize = lambda s: s
    print("WARNING: normalizer not found, running without it (09-তে থাকলে এখানেও থাকা উচিত)")

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:2000].map(bn_normalize)
    df["domain"] = df["domain"].fillna("unk")
    df["label"] = df["label"].astype(int)

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=False)

class TxtDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAXLEN, padding=False)
        self.labels = df["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def fit_scores(tr, te):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    vc = tr["label"].value_counts()
    cw = torch.tensor([len(tr)/(2*vc.get(0,1)), len(tr)/(2*vc.get(1,1))], dtype=torch.float)
    args = TrainingArguments(
        output_dir="/tmp/bb_gk", per_device_train_batch_size=16, per_device_eval_batch_size=64,
        num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01, logging_steps=200,
        save_strategy="no", report_to=[], seed=SEED, fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(model=model, args=args, train_dataset=TxtDS(tr),
                              data_collator=lambda f: tok.pad(f, return_tensors="pt"),
                              class_weights=cw)
    trainer.train()
    logits = trainer.predict(TxtDS(te)).predictions
    del trainer, model; torch.cuda.empty_cache()
    return logits[:, 1] - logits[:, 0]

def run_groupkfold(name):
    df = CORP[name]
    for fold, (tr_i, te_i) in enumerate(GroupKFold(n_splits=5).split(df, groups=df["domain"])):
        f_s, f_i = f"{OUT}/{name}_fold{fold}_scores.npy", f"{OUT}/{name}_fold{fold}_idx.npy"
        if os.path.exists(f_s):
            print(f"[{name}] fold {fold} আগেই শেষ, skip"); continue
        print(f"[{name}] fold {fold}: train={len(tr_i)} test={len(te_i)}")
        s = fit_scores(df.iloc[tr_i], df.iloc[te_i])
        np.save(f_i, te_i); np.save(f_s, s)

def pooled(name):
    df = CORP[name]; ys, ss = [], []
    for fold in range(5):
        te_i = np.load(f"{OUT}/{name}_fold{fold}_idx.npy"); s = np.load(f"{OUT}/{name}_fold{fold}_scores.npy")
        ys.append(df.iloc[te_i]["label"].values); ss.append(s)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    r = dict(corpus=name, model="BanglaBERT", n=len(y),
             bacc=round(balanced_accuracy_score(y, p), 3),
             recall0=round(((p==0)&(y==0)).sum()/(y==0).sum(), 3),
             recall1=round(((p==1)&(y==1)).sum()/(y==1).sum(), 3),
             auc=round(roc_auc_score(y, s), 3))
    pd.DataFrame([r]).to_csv(f"{R}/groupkfold_banglabert_{name}.csv", index=False)
    return r
print("setup ok | GPU:", torch.cuda.is_available())

setup ok | GPU: True


In [2]:
run_groupkfold("c2025")
print(pooled("c2025"))

[c2025] fold 0: train=2562 test=1417


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.233100
400,0.110900


[c2025] fold 1: train=3347 test=632


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Step,Training Loss
200,0.189800
400,0.042800
600,0.031500


[c2025] fold 4: train=3336 test=643


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.185700
400,0.043400
600,0.031200


{'corpus': 'c2025', 'model': 'BanglaBERT', 'n': 3979, 'bacc': 0.711, 'recall0': np.float64(0.435), 'recall1': np.float64(0.987), 'auc': 0.979}


In [3]:
run_groupkfold("c2020")
print(pooled("c2020"))

[c2020] fold 0: train=39869 test=9952


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.439000
400,0.255700
600,0.241400
800,0.298100
1000,0.223800
1200,0.175100
1400,0.255200
1600,0.142700
1800,0.285200
2000,0.211900


[c2020] fold 1: train=39869 test=9952


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Step,Training Loss
200,0.591500
400,0.473900
600,0.394200
800,0.344800
1000,0.310300
1200,0.329700
1400,0.309000
1600,0.314800
1800,0.237800
2000,0.170600


[c2020] fold 4: train=39808 test=10013


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.529200
400,0.370600
600,0.270000
800,0.283400
1000,0.242300
1200,0.168900
1400,0.215100
1600,0.288600
1800,0.179800
2000,0.144800


{'corpus': 'c2020', 'model': 'BanglaBERT', 'n': 49821, 'bacc': 0.794, 'recall0': np.float64(0.591), 'recall1': np.float64(0.998), 'auc': 0.944}


In [4]:
# 12 — Cell 4: diagnostic only (per-fold, scores already saved; no training)
df = CORP["c2025"]
for fold in range(5):
    te_i = np.load(f"{OUT}/c2025_fold{fold}_idx.npy"); s = np.load(f"{OUT}/c2025_fold{fold}_scores.npy")
    y = df.iloc[te_i]["label"].values; p = (s > 0).astype(int)
    auc = roc_auc_score(y, s) if len(set(y)) == 2 else float("nan")
    print(f"fold {fold} | n={len(y)} fake={int((y==0).sum())} | "
          f"recall0={((p==0)&(y==0)).sum()/max((y==0).sum(),1):.2f} "
          f"recall1={((p==1)&(y==1)).sum()/max((y==1).sum(),1):.2f} | AUC={auc:.3f} | median score={np.median(s):+.2f}")

fold 0 | n=1417 fake=1417 | recall0=0.51 recall1=0.00 | AUC=nan | median score=-0.14
fold 1 | n=632 fake=19 | recall0=0.21 recall1=0.99 | AUC=0.786 | median score=+5.61
fold 2 | n=632 fake=538 | recall0=0.25 recall1=1.00 | AUC=0.971 | median score=+3.88
fold 3 | n=655 fake=5 | recall0=0.00 recall1=0.99 | AUC=0.872 | median score=+5.52
fold 4 | n=643 fake=6 | recall0=0.17 recall1=0.98 | AUC=0.836 | median score=+5.66


In [5]:
# 12 — Cell 5: label semantics check (no training, ~seconds)
import pandas as pd
P = "/mnt/g/banglafake-detection/data/processed/v2"
df = pd.read_csv(f"{P}/c2025.csv")
df["domain"] = df["domain"].fillna("unk")

# (ক) ডোমেইন তালিকা: সাইজ ও label
g = df.groupby("domain")["label"].agg(rows="size", label_mean="mean").sort_values("rows", ascending=False)
print(g.round(3).to_string())

# (খ) label-0 ডোমেইনের নমুনা: প্রতি ডোমেইন থেকে ২টি, প্রথম ২০০ অক্ষর
print("\n=== label-0 samples ===")
for d in g[g["label_mean"] < 0.5].index:
    sub = df[(df["domain"] == d) & (df["label"] == 0)]
    for t in sub["text"].fillna("").sample(min(2, len(sub)), random_state=0):
        print(f"[{d}] {t[:200]!r}\n")

                     rows  label_mean
domain                               
Facebook Page        1417       0.000
Prothomalo            599       0.980
Earki.com             527       0.000
Samakal News          363       0.997
Ittefaq               322       0.991
News 24               181       0.983
Bangladesh Pratidin   156       0.994
Independent News      140       1.000
Jamuna TV             136       0.978
Independent news       88       0.966
Dhakatribune           17       1.000
BBC News                9       1.000
Amar Sangbad            8       1.000
Somoy News              4       0.000
Deshrupantor            1       0.000
Dhaka Post              1       0.000
Anandabazar             1       0.000
Kaler Kantho            1       0.000
Prothom Alo             1       0.000
News24                  1       0.000
NTV                     1       0.000
Samakal                 1       1.000
Viral News              1       0.000
Worthy Talk BD          1       0.000
dakshinancha

In [6]:
# 12 — Cell 6: domain-name normalization vs GroupKFold folds (no training)
import re, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
df = pd.read_csv(f"{P}/c2025.csv"); df["domain"] = df["domain"].fillna("unk")
norm = lambda d: {"samakalnews": "samakal"}.get(re.sub(r"[^a-z0-9]", "", d.lower()),
                                                re.sub(r"[^a-z0-9]", "", d.lower()))
df["domain_n"] = df["domain"].map(norm)
print("raw domains:", df["domain"].nunique(), "| normalized:", df["domain_n"].nunique())
def fold_of(g):
    f = np.empty(len(df), int)
    for k, (_, te) in enumerate(GroupKFold(5).split(df, groups=g)): f[te] = k
    return f
print(pd.crosstab(fold_of(df["domain"]), fold_of(df["domain_n"])))

raw domains: 26 | normalized: 22
col_0     0    1    2    3    4
row_0                          
0      1417    0    0    0    0
1         0  599    0    2   31
2         0    1  527    0  104
3         0    0  156  499    0
4         0    0    0  181  462


In [7]:
# 12 — Cell 6b: c2020 normalization effect on folds (no training)
import re
def norm(d):
    k = re.sub(r"[^a-z0-9]", "", d.lower()); return {"samakalnews": "samakal"}.get(k, k)
d = CORP["c2020"].copy(); d["domain_n"] = d["domain"].map(norm)
print("c2020 raw:", d["domain"].nunique(), "| normalized:", d["domain_n"].nunique())
def fold_of(col):
    f = np.empty(len(d), int)
    for k, (_, te) in enumerate(GroupKFold(5).split(d, groups=d[col])): f[te] = k
    return f
ct = pd.crosstab(fold_of("domain"), fold_of("domain_n")); print(ct)
print("moved rows:", len(d) - int(ct.max(axis=1).sum()), "of", len(d))

c2020 raw: 93 | normalized: 92
col_0     0     1     2     3      4
row_0                               
0      9952     0     0     0      0
1         0  9938    14     0      0
2         0    14  9938     0      0
3         0     0     0  9952      0
4         0     0     0     0  10013
moved rows: 28 of 49821


In [8]:
# 12 — Cell 7: c2025 rerun with normalized domain groups (new output folder)
import re
def norm(d):
    k = re.sub(r"[^a-z0-9]", "", d.lower()); return {"samakalnews": "samakal"}.get(k, k)
CORP["c2025"]["domain_n"] = CORP["c2025"]["domain"].map(norm)
OUT_N = f"{R}/bb_groupkfold_norm"; os.makedirs(OUT_N, exist_ok=True)

def run_gk_norm(name):
    df = CORP[name]
    for fold, (tr_i, te_i) in enumerate(GroupKFold(5).split(df, groups=df["domain_n"])):
        f_s = f"{OUT_N}/{name}_fold{fold}_scores.npy"
        if os.path.exists(f_s): print(f"fold {fold} skip"); continue
        print(f"[{name}] fold {fold}: train={len(tr_i)} test={len(te_i)}")
        np.save(f"{OUT_N}/{name}_fold{fold}_idx.npy", te_i)
        np.save(f_s, fit_scores(df.iloc[tr_i], df.iloc[te_i]))

def pooled_norm(name):
    df = CORP[name]; ys, ss = [], []
    for fold in range(5):
        te_i = np.load(f"{OUT_N}/{name}_fold{fold}_idx.npy"); s = np.load(f"{OUT_N}/{name}_fold{fold}_scores.npy")
        ys.append(df.iloc[te_i]["label"].values); ss.append(s)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    r = dict(corpus=name, model="BanglaBERT_normdomain", n=len(y),
             bacc=round(balanced_accuracy_score(y, p), 3),
             recall0=round(((p==0)&(y==0)).sum()/(y==0).sum(), 3),
             recall1=round(((p==1)&(y==1)).sum()/(y==1).sum(), 3),
             auc=round(roc_auc_score(y, s), 3))
    pd.DataFrame([r]).to_csv(f"{R}/groupkfold_banglabert_{name}_norm.csv", index=False)
    return r

run_gk_norm("c2025"); print(pooled_norm("c2025"))

[c2025] fold 0: train=2562 test=1417


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.240800
400,0.109800


[c2025] fold 1: train=3379 test=600


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Step,Training Loss
200,0.179400
400,0.051600
600,0.027700


[c2025] fold 4: train=3382 test=597


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.181300
400,0.053200
600,0.027300


{'corpus': 'c2025', 'model': 'BanglaBERT_normdomain', 'n': 3979, 'bacc': 0.646, 'recall0': np.float64(0.306), 'recall1': np.float64(0.987), 'auc': 0.975}


In [9]:
# 12 — Cell 8: extra seeds (1, 2) on normalized c2025 folds
import re
def norm(d):
    k = re.sub(r"[^a-z0-9]", "", d.lower()); return {"samakalnews": "samakal"}.get(k, k)
CORP["c2025"]["domain_n"] = CORP["c2025"]["domain"].map(norm)

def run_seed(name, seed):
    global SEED
    SEED = seed
    out = f"{R}/bb_groupkfold_norm_seed{seed}"; os.makedirs(out, exist_ok=True)
    df = CORP[name]
    for fold, (tr_i, te_i) in enumerate(GroupKFold(5).split(df, groups=df["domain_n"])):
        f_s = f"{out}/{name}_fold{fold}_scores.npy"
        if os.path.exists(f_s): print(f"seed {seed} fold {fold} skip"); continue
        print(f"[{name}] seed {seed} fold {fold}: train={len(tr_i)} test={len(te_i)}")
        np.save(f"{out}/{name}_fold{fold}_idx.npy", te_i)
        np.save(f_s, fit_scores(df.iloc[tr_i], df.iloc[te_i]))

def pooled_dir(name, out, tag):
    df = CORP[name]; ys, ss = [], []
    for fold in range(5):
        te_i = np.load(f"{out}/{name}_fold{fold}_idx.npy"); s = np.load(f"{out}/{name}_fold{fold}_scores.npy")
        ys.append(df.iloc[te_i]["label"].values); ss.append(s)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    return dict(seed=tag,
                bacc=balanced_accuracy_score(y, p),
                recall0=((p==0)&(y==0)).sum()/(y==0).sum(),
                recall1=((p==1)&(y==1)).sum()/(y==1).sum(),
                auc=roc_auc_score(y, s))

EXTRA = [1, 2]
for sd in EXTRA:
    run_seed("c2025", sd)

rows = [pooled_dir("c2025", f"{R}/bb_groupkfold_norm", 0)]
rows += [pooled_dir("c2025", f"{R}/bb_groupkfold_norm_seed{sd}", sd) for sd in EXTRA]
t = pd.DataFrame(rows)
print(t.round(3).to_string(index=False))
print(t[["bacc", "recall0", "recall1", "auc"]].agg(["mean", "std"]).round(3))
t.to_csv(f"{R}/groupkfold_banglabert_c2025_norm_seeds.csv", index=False)

[c2025] seed 1 fold 0: train=2562 test=1417


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.236000
400,0.089000


[c2025] seed 1 fold 1: train=3379 test=600


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Step,Training Loss
200,0.185600
400,0.040200
600,0.019600


[c2025] seed 2 fold 4: train=3382 test=597


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.198200
400,0.049000
600,0.031500


 seed  bacc  recall0  recall1   auc
    0 0.646    0.306    0.987 0.975
    1 0.737    0.489    0.984 0.976
    2 0.693    0.398    0.987 0.979
       bacc  recall0  recall1    auc
mean  0.692    0.398    0.986  0.977
std   0.045    0.092    0.002  0.002


In [10]:
import pandas as pd
P = "/mnt/g/banglafake-detection/data/processed/v2"
df = pd.read_csv(f"{P}/c2025.csv")
df["text"] = df["text"].fillna(""); df["domain"] = df["domain"].fillna("unk")
fb = df[df["domain"] == "Facebook Page"]
for t in fb["text"].sample(10, random_state=1):
    print(repr(t[:220]), "\n")
for name, sub in [("Facebook Page", fb), ("Earki.com", df[df["domain"] == "Earki.com"]),
                  ("real (label 1)", df[df["label"] == 1])]:
    print(name, len(sub), "| contains 'শীর্ষক':", round(sub["text"].str.contains("শীর্ষক").mean(), 3))

'বিএসএফ সদস্য আটককে ঘিরে কিংবা কাশ্মীর সীমান্তে ভারত-পাকিস্তান সংঘর্ষে ৮৫ জনের প্রাণহানি ভারত-শাসিত কাশ্মীরে গত ২২ এপ্রিল সশস্ত্র গোষ্ঠীর ভয়াবহ হামলাকে কেন্দ্র করে ভারত ও পাকিস্তানের মধ্যে উত্তেজনা চরম পর্যায়ে রয়েছে। এ' 

'শহীদ ওসমান হাদি হ’ত্যার বিচার না হওয়া পর্যন্ত দেশে কোন নির্বাচন হবে না- ভিপি সাদিক কায়েম শরিফ ওসমান হাদি মৃত্যুবরণ করেন এই ঘটনার প্রেক্ষিতে সম্প্রতি ‘শহীদ ওসমান হাদি হ’ত্যার বিচার না হওয়া পর্যন্ত দেশে কোন নির্বাচন হবে' 

'উপবৃত্তি বাদেও দেওয়া হবে স্টুডেন্ট ভাতা: ড. ইউনুস প্রধান উপদেষ্টা সম্প্রতি, “উপবৃত্তি বাদেও দেওয়া হবে স্টুডেন্ট ভাতা: ড. ইউনুস প্রধান উপদেষ্টা’’ শীর্ষক শিরোনামে বেসরকারি ইলেকট্রনিক সংবাদমাধ্যম যমুনা টেলিভিশন এর ডিজাইন ' 

'ব্যাংক কর্মকর্তাকে রাত ১০ টার মধ্যে ঘুমানোর নির্দেশের চিঠি রাত ১০টার মধ্যে ঘুমাতে ব্যাংক কর্মকর্তাকে চিঠি” শীর্ষক শিরোনামে একটি তথ্যসম্বলিত চিঠি সামাজিক যোগাযোগ মাধ্যম ফেসবুকে ছড়িয়ে পড়েছে।' 

'মুমূর্ষু খালেদা জিয়ার শেষ রাজনৈতিক বাণী মুমূর্ষু খালেদা জিয়ার শেষ রাজনৈতিক বাণী-শিরোনামে একটি ফটোকার্ড সামাজিক মাধ্যমে ছড়িয়ে পড়ে

In [11]:
import os, re, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"

def norm(d):
    k = re.sub(r"[^a-z0-9]", "", d.lower()); return {"samakalnews": "samakal"}.get(k, k)

PAT = [r"শীর্ষক", r"শিরোনামে?", r"দাবিতে?", r"সামাজিক\s+(?:যোগাযোগ\s+)?মাধ্যমে?",
       r"ফেসবুকে?", r"ছড়িয়ে\s+(?:পড়েছে|পড়ছে|দেওয়া|দেয়া)",
       r"প্রচার(?:িত|\s+করা\s+হচ্ছে|\s+করেছে)", r"তথ্য\s*সম্বলিত",
       r"ফটোকার্ড", r"স্ক্রিনশট", r"ডিজিটাল\s+ব্যানার", r"ভাইরাল", r"[“”‘’\"]"]
FC = re.compile("|".join(PAT))

df = pd.read_csv(f"{P}/c2025.csv")
df["text"] = df["text"].fillna("").str[:1500]
df["domain_raw"] = df["domain"].fillna("unk")
df["domain"] = df["domain_raw"].map(norm)
df["label"] = df["label"].astype(int)
df["text_masked"] = df["text"].map(lambda t: re.sub(r"\s+", " ", FC.sub(" ", t)).strip())

grp = np.where(df["domain_raw"] == "Facebook Page", "Facebook Page",
      np.where(df["domain_raw"] == "Earki.com", "Earki",
      np.where(df["label"] == 1, "real", "other fake")))
chg = (df["text"] != df["text_masked"]).values
print("rows changed by masking:", {g: round(float(chg[grp == g].mean()), 3) for g in np.unique(grp)})

def fit_scores(tr, te, col):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(v.fit_transform(tr[col]), tr["label"])
    return clf.decision_function(v.transform(te[col]))

def summarize(y, s, dom):
    p = (s > 0).astype(int)
    out = dict(bacc=balanced_accuracy_score(y, p),
               recall0=((p == 0) & (y == 0)).sum() / (y == 0).sum(),
               recall1=((p == 1) & (y == 1)).sum() / (y == 1).sum(),
               auc=roc_auc_score(y, s))
    for g, k in (("Facebook Page", "rec0_FB"), ("Earki.com", "rec0_Earki")):
        m = dom == g
        out[k] = ((p == 0) & m).sum() / m.sum()
    return out

rows = []
for col in ("text", "text_masked"):
    ys, ss, ds = [], [], []
    for tr_i, te_i in GroupKFold(5).split(df, groups=df["domain"]):
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        ss.append(fit_scores(tr, te, col)); ys.append(te["label"].values); ds.append(te["domain_raw"].values)
    rows.append(dict(setting="GroupKFold", text=col,
                     **summarize(np.concatenate(ys), np.concatenate(ss), np.concatenate(ds))))
    for seed in (0, 1, 2):
        tr, te = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=seed)
        s = fit_scores(tr, te, col); y = te["label"].values
        rows.append(dict(setting=f"random_seed{seed}", text=col,
                         bacc=balanced_accuracy_score(y, (s > 0).astype(int)), auc=roc_auc_score(y, s)))
res = pd.DataFrame(rows).round(3)
res.to_csv(f"{R}/factcheck_phrase_masking.csv", index=False)
print(res.to_string(index=False))

rows changed by masking: {np.str_('Earki'): 0.727, np.str_('Facebook Page'): 0.936, np.str_('other fake'): 0.537, np.str_('real'): 0.517}
     setting        text  bacc  recall0  recall1   auc  rec0_FB  rec0_Earki
  GroupKFold        text 0.505    0.041    0.969 0.678    0.047       0.009
random_seed0        text 0.960      NaN      NaN 0.992      NaN         NaN
random_seed1        text 0.951      NaN      NaN 0.991      NaN         NaN
random_seed2        text 0.960      NaN      NaN 0.990      NaN         NaN
  GroupKFold text_masked 0.500    0.038    0.961 0.653    0.040       0.015
random_seed0 text_masked 0.956      NaN      NaN 0.991      NaN         NaN
random_seed1 text_masked 0.952      NaN      NaN 0.990      NaN         NaN
random_seed2 text_masked 0.961      NaN      NaN 0.990      NaN         NaN
